# 04. Hu/TransRAC JSON and NPZ Features

This notebook generates Hu/TransRAC-style JSON feature files from the validated 64-frame pose dataset, validates the JSON outputs, converts them to NPZ, and validates the NPZ files.


## 1. Import Libraries and Set Paths

The input folder is `data/resized_interpolated_json`.

The JSON output folder is `data/hu_transrac_json`.

The NPZ output folder is `data/hu_transrac_npz`.


In [ ]:
from pathlib import Path
import json
import math
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
from tqdm import tqdm

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

LOCAL_PROJECT_ROOT = Path(
    "/Users/jhonmichaelferrer/Documents/VSCode/Coreset/CoreSet-AI-Fitness-Tracker-ML-Pipeline"
)
if not (PROJECT_ROOT / "data").exists() and LOCAL_PROJECT_ROOT.exists():
    PROJECT_ROOT = LOCAL_PROJECT_ROOT

INPUT_BASE = PROJECT_ROOT / "data" / "resized_interpolated_json"
JSON_OUTPUT_BASE = PROJECT_ROOT / "data" / "hu_transrac_json"
NPZ_OUTPUT_BASE = PROJECT_ROOT / "data" / "hu_transrac_npz"

CATEGORIES = ["bench_press", "pull_up", "push_up", "squat"]
CATEGORY_TO_INDEX = {category: idx for idx, category in enumerate(CATEGORIES)}

EXPECTED_FRAMES = 64
EXPECTED_LANDMARKS = 33
FEATURES = 4
DENSITY_SIGMA = 2.0

print("Project root:", PROJECT_ROOT)
print("Input folder exists:", INPUT_BASE.exists())
print("JSON output folder:", JSON_OUTPUT_BASE)
print("NPZ output folder:", NPZ_OUTPUT_BASE)

for category in CATEGORIES:
    category_path = INPUT_BASE / category
    files = sorted(category_path.glob("*.json"))
    print(
        f"{category}: "
        f"folder_exists={category_path.exists()} "
        f"json_files={len(files)}"
    )


## 2. Load a 64-Frame Pose Sequence

This function converts one resized JSON file into a NumPy pose array with shape `64 x 33 x 4`.


In [ ]:
def load_pose_array(frames):
    pose = np.zeros((EXPECTED_FRAMES, EXPECTED_LANDMARKS, FEATURES), dtype=np.float64)

    if len(frames) != EXPECTED_FRAMES:
        raise ValueError(f"Expected {EXPECTED_FRAMES} frames, found {len(frames)}")

    for frame_idx, frame in enumerate(frames):
        landmarks = frame.get("landmarks")

        if not isinstance(landmarks, list) or len(landmarks) != EXPECTED_LANDMARKS:
            raise ValueError(f"Frame {frame_idx} has invalid landmarks")

        for landmark_idx, landmark in enumerate(landmarks):
            if not isinstance(landmark, list) or len(landmark) != FEATURES:
                raise ValueError(f"Frame {frame_idx} landmark {landmark_idx} is invalid")

            values = [float(value) for value in landmark]
            if any(not math.isfinite(value) for value in values):
                raise ValueError(f"Frame {frame_idx} landmark {landmark_idx} has non-finite values")

            pose[frame_idx, landmark_idx] = values

    pose[:, :, 3] = np.clip(pose[:, :, 3], 0.0, 1.0)
    return pose


## 3. Normalize the Pose Sequence

This function centers each frame using visibility-weighted landmark coordinates and scales it using shoulder distance.


In [ ]:
def normalize_pose(pose):
    normalized = pose.copy()
    coords = normalized[:, :, :3]
    visibility = normalized[:, :, 3:4]

    weighted = coords * visibility
    visible_sum = np.maximum(visibility.sum(axis=1, keepdims=True), 1e-8)
    center = weighted.sum(axis=1, keepdims=True) / visible_sum
    coords = coords - center

    scale = np.linalg.norm(coords[:, 11, :2] - coords[:, 12, :2], axis=1)
    fallback_scale = np.nanmedian(scale[scale > 1e-8]) if np.any(scale > 1e-8) else 1.0
    scale = np.where(scale > 1e-8, scale, fallback_scale).reshape(-1, 1, 1)

    normalized[:, :, :3] = coords / scale
    return normalized


## 4. Compute the Motion Signal

This function measures frame-to-frame pose movement and produces a 64-value motion signal.


In [ ]:
def compute_motion_signal(normalized_pose):
    coords = normalized_pose[:, :, :3]
    visibility = normalized_pose[:, :, 3]

    deltas = np.linalg.norm(np.diff(coords, axis=0), axis=2)
    weights = np.minimum(visibility[1:], visibility[:-1])
    weighted_motion = (deltas * weights).sum(axis=1) / np.maximum(weights.sum(axis=1), 1e-8)

    motion = np.concatenate([[0.0], weighted_motion])

    if motion.max() > motion.min():
        motion = (motion - motion.min()) / (motion.max() - motion.min())
    else:
        motion = np.zeros_like(motion)

    if len(motion) >= 5:
        kernel = np.array([1, 2, 3, 2, 1], dtype=np.float64)
        kernel = kernel / kernel.sum()
        motion = np.convolve(motion, kernel, mode="same")

    return motion


## 5. Estimate Repetition Time Points

This function detects strong local peaks in the motion signal and stores them as `time_points`.


In [ ]:
def estimate_time_points(motion):
    if len(motion) == 0:
        return []

    threshold = float(max(0.15, motion.mean() + 0.5 * motion.std()))
    candidates = []

    for idx in range(1, len(motion) - 1):
        if motion[idx] >= motion[idx - 1] and motion[idx] >= motion[idx + 1] and motion[idx] >= threshold:
            candidates.append(idx)

    min_distance = 6
    selected = []

    for idx in sorted(candidates, key=lambda i: motion[i], reverse=True):
        if all(abs(idx - existing) >= min_distance for existing in selected):
            selected.append(idx)

    selected.sort()

    if not selected:
        selected = [int(np.argmax(motion))]

    return [int(idx) for idx in selected]


## 6. Create a Density Map

This function converts the detected `time_points` into a 64-value density map.

The density map sum is scaled to match the estimated repetition count.


In [ ]:
def make_density_map(time_points, frame_count=EXPECTED_FRAMES, sigma=DENSITY_SIGMA):
    x = np.arange(frame_count, dtype=np.float64)
    density = np.zeros(frame_count, dtype=np.float64)

    for point in time_points:
        density += np.exp(-0.5 * ((x - point) / sigma) ** 2)

    total = density.sum()
    if total > 0:
        density *= len(time_points) / total

    return density


## 7. Generate One Hu/TransRAC JSON File

This function processes one resized pose JSON file and writes one Hu/TransRAC-style JSON feature file.


In [ ]:
def process_json_file(json_path):
    relative_path = json_path.relative_to(INPUT_BASE)
    category = relative_path.parts[0]
    output_path = JSON_OUTPUT_BASE / relative_path
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with json_path.open("r") as f:
        data = json.load(f)

    pose = load_pose_array(data.get("frames", []))
    normalized_pose = normalize_pose(pose)
    motion_signal = compute_motion_signal(normalized_pose)
    time_points = estimate_time_points(motion_signal)
    density_map = make_density_map(time_points)

    output_data = {
        "source_json": str(relative_path),
        "video_path": data.get("video_path"),
        "fps": data.get("fps"),
        "category": category,
        "category_index": CATEGORY_TO_INDEX[category],
        "frame_count": EXPECTED_FRAMES,
        "landmark_count": EXPECTED_LANDMARKS,
        "feature_count": FEATURES,
        "original_frame_count": data.get("original_frame_count"),
        "resized_frame_count": data.get("resized_frame_count"),
        "resize_method": data.get("resize_method"),
        "feature_method": "hu_transrac_style_pose_density_json_from_64_frame_resized_pose",
        "pose_sequence": pose.tolist(),
        "normalized_pose_sequence": normalized_pose.tolist(),
        "motion_signal": motion_signal.tolist(),
        "density_map": density_map.tolist(),
        "density_sum": float(density_map.sum()),
        "time_points": time_points,
        "rep_count": len(time_points),
    }

    with output_path.open("w") as f:
        json.dump(output_data, f, separators=(",", ":"))

    return category


## 8. Generate Hu/TransRAC JSON for All Files

This cell processes all 1000 resized JSON files and writes the Hu/TransRAC JSON outputs.


In [ ]:
json_paths = []

for category in CATEGORIES:
    json_paths.extend(sorted((INPUT_BASE / category).glob("*.json")))

if not json_paths:
    raise FileNotFoundError(f"No JSON files found in {INPUT_BASE}")

category_counts = Counter()
errors = []

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(process_json_file, path): path for path in json_paths}

    for future in tqdm(as_completed(futures), total=len(futures), desc="hu_transrac_json"):
        path = futures[future]
        try:
            category_counts[future.result()] += 1
        except Exception as exc:
            errors.append((str(path.relative_to(INPUT_BASE)), str(exc)))

print("completed", sum(category_counts.values()))
print("category_counts", dict(category_counts))
print("errors", len(errors))

if errors:
    print("first_errors")
    for path, error in errors[:10]:
        print("-", path, error)

print("output", JSON_OUTPUT_BASE)


## 9. Set Hu/TransRAC JSON Validation Rules

This cell prepares the validation containers for the generated JSON files.


In [ ]:
BASE = JSON_OUTPUT_BASE

category_counts = Counter()
stats = Counter()
rep_counts = Counter()
time_point_lengths = Counter()
errors = []
samples = []

json_paths = sorted(BASE.rglob("*.json"))

print("Validation folder:", BASE)
print("JSON files found:", len(json_paths))

for category in CATEGORIES:
    category_path = BASE / category
    files = sorted(category_path.glob("*.json"))
    print(
        f"{category}: "
        f"folder_exists={category_path.exists()} "
        f"json_files={len(files)}"
    )


## 10. Validate Hu/TransRAC JSON Contents

This cell validates JSON keys, pose shapes, motion signals, density maps, time points, and repetition-count consistency.


In [ ]:
for path in json_paths:
    rel = path.relative_to(BASE)
    category = rel.parts[0] if len(rel.parts) > 1 else "unknown"

    category_counts[category] += 1
    stats["files"] += 1

    try:
        with path.open("r") as f:
            data = json.load(f)
    except Exception as exc:
        errors.append((str(rel), f"json_load_error: {exc}"))
        continue

    required = [
        "source_json", "video_path", "fps", "category", "category_index",
        "frame_count", "landmark_count", "feature_count", "feature_method",
        "pose_sequence", "normalized_pose_sequence", "motion_signal",
        "density_map", "density_sum", "time_points", "rep_count",
    ]

    for key in required:
        if key not in data:
            errors.append((str(rel), f"missing key {key}"))

    if data.get("category") != category:
        errors.append((str(rel), f"category mismatch {data.get('category')} != {category}"))

    pose = data.get("pose_sequence")
    normalized = data.get("normalized_pose_sequence")
    motion = data.get("motion_signal")
    density = data.get("density_map")
    time_points = data.get("time_points")
    rep_count = data.get("rep_count")

    for name, sequence in [("pose_sequence", pose), ("normalized_pose_sequence", normalized)]:
        if not isinstance(sequence, list) or len(sequence) != EXPECTED_FRAMES:
            errors.append((str(rel), f"{name} bad frame length"))
            continue

        for frame_idx, frame in enumerate(sequence):
            if not isinstance(frame, list) or len(frame) != EXPECTED_LANDMARKS:
                errors.append((str(rel), f"{name} frame {frame_idx} bad landmark length"))
                break

            for landmark_idx, landmark in enumerate(frame):
                if not isinstance(landmark, list) or len(landmark) != FEATURES:
                    errors.append((str(rel), f"{name} frame {frame_idx} landmark {landmark_idx} bad values"))
                    break

                values = [float(value) for value in landmark]
                if any(not math.isfinite(value) for value in values):
                    errors.append((str(rel), f"{name} frame {frame_idx} landmark {landmark_idx} nonfinite"))
                    break

    if not isinstance(motion, list) or len(motion) != EXPECTED_FRAMES:
        errors.append((str(rel), "motion_signal bad length"))
    elif any(not math.isfinite(float(value)) for value in motion):
        errors.append((str(rel), "motion_signal nonfinite"))

    if not isinstance(density, list) or len(density) != EXPECTED_FRAMES:
        errors.append((str(rel), "density_map bad length"))
    elif any(not math.isfinite(float(value)) for value in density):
        errors.append((str(rel), "density_map nonfinite"))

    if not isinstance(time_points, list):
        errors.append((str(rel), "time_points not list"))
    else:
        time_point_lengths[len(time_points)] += 1
        for point in time_points:
            if not isinstance(point, int) or point < 0 or point >= EXPECTED_FRAMES:
                errors.append((str(rel), f"invalid time_point {point}"))

    if not isinstance(rep_count, int):
        errors.append((str(rel), "rep_count not int"))
    else:
        rep_counts[rep_count] += 1
        if isinstance(time_points, list) and rep_count != len(time_points):
            errors.append((str(rel), "rep_count does not match time_points length"))
        if isinstance(density, list) and abs(float(data.get("density_sum", -999)) - rep_count) > 1e-6:
            errors.append((str(rel), "density_sum does not match rep_count"))

    if len(samples) < 2:
        samples.append({
            "path": str(rel),
            "category": data.get("category"),
            "frame_count": data.get("frame_count"),
            "pose_shape": [len(pose), len(pose[0]), len(pose[0][0])] if isinstance(pose, list) else None,
            "time_points": time_points,
            "rep_count": rep_count,
            "density_sum": data.get("density_sum"),
        })


## 11. Print Hu/TransRAC JSON Validation Report

This cell prints a formatted validation report for the generated JSON files.


In [ ]:
print("=" * 78)
print("HU TRANSRAC JSON VALIDATION REPORT".center(78))
print("=" * 78)
print(f"{'Category':<20} | {'Files':<6}")
print("-" * 78)

for category in CATEGORIES:
    print(f"{category:<20} | {category_counts[category]:<6}")

print("-" * 78)
print(f"{'TOTAL':<20} | {stats['files']:<6}")
print("=" * 78)

print()
print("REPETITION COUNT DISTRIBUTION")
print("-" * 78)
print(f"{'Rep Count':<12} | {'Files':<6}")
print("-" * 78)
for rep_count, file_count in sorted(rep_counts.items()):
    print(f"{rep_count:<12} | {file_count:<6}")
print("-" * 78)

print()
print("TIME POINTS LENGTH DISTRIBUTION")
print("-" * 78)
print(f"{'Time Points':<12} | {'Files':<6}")
print("-" * 78)
for time_point_count, file_count in sorted(time_point_lengths.items()):
    print(f"{time_point_count:<12} | {file_count:<6}")
print("-" * 78)

print()
print("VALIDATION SUMMARY")
print("-" * 78)
validation_ok = (
    stats["files"] == 1000
    and all(category_counts[category] == 250 for category in CATEGORIES)
    and len(errors) == 0
)
print(f"{'Errors':<25}: {len(errors)}")
print(f"{'Validation OK':<25}: {validation_ok}")

print()
print("SAMPLE OUTPUTS")
print("-" * 78)
print(
    f"{'Path':<45} | {'Category':<12} | {'Frames':<6} | "
    f"{'Shape':<12} | {'Time Points':<15} | {'Rep':<4} | {'Density Sum':<11}"
)
print("-" * 78)

for sample in samples:
    shape = "x".join(str(value) for value in sample["pose_shape"])
    time_points = str(sample["time_points"])
    path_text = sample["path"][:42] + "..." if len(sample["path"]) > 45 else sample["path"]
    print(
        f"{path_text:<45} | {sample['category']:<12} | "
        f"{sample['frame_count']:<6} | {shape:<12} | "
        f"{time_points:<15} | {sample['rep_count']:<4} | "
        f"{sample['density_sum']:<11.2f}"
    )
print("-" * 78)

if errors:
    print()
    print("FIRST ERRORS")
    print("-" * 78)
    for path, error in errors[:25]:
        print(f"{path}: {error}")


## 12. Convert Hu/TransRAC JSON Files to NPZ

This cell converts each validated Hu/TransRAC JSON file into a compressed `.npz` file.


In [ ]:
def convert_json_to_npz(json_path):
    relative_path = json_path.relative_to(JSON_OUTPUT_BASE)
    output_path = (NPZ_OUTPUT_BASE / relative_path).with_suffix(".npz")
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with json_path.open("r") as f:
        data = json.load(f)

    pose_sequence = np.asarray(data["pose_sequence"], dtype=np.float32)
    normalized_pose_sequence = np.asarray(data["normalized_pose_sequence"], dtype=np.float32)
    motion_signal = np.asarray(data["motion_signal"], dtype=np.float32)
    density_map = np.asarray(data["density_map"], dtype=np.float32)
    time_points = np.asarray(data["time_points"], dtype=np.int64)

    if pose_sequence.shape != (EXPECTED_FRAMES, EXPECTED_LANDMARKS, FEATURES):
        raise ValueError(f"bad pose_sequence shape {pose_sequence.shape}")
    if normalized_pose_sequence.shape != (EXPECTED_FRAMES, EXPECTED_LANDMARKS, FEATURES):
        raise ValueError(f"bad normalized_pose_sequence shape {normalized_pose_sequence.shape}")
    if motion_signal.shape != (EXPECTED_FRAMES,):
        raise ValueError(f"bad motion_signal shape {motion_signal.shape}")
    if density_map.shape != (EXPECTED_FRAMES,):
        raise ValueError(f"bad density_map shape {density_map.shape}")

    np.savez_compressed(
        output_path,
        pose_sequence=pose_sequence,
        normalized_pose_sequence=normalized_pose_sequence,
        motion_signal=motion_signal,
        density_map=density_map,
        time_points=time_points,
        rep_count=np.asarray(data["rep_count"], dtype=np.int64),
        category_index=np.asarray(data["category_index"], dtype=np.int64),
        frame_count=np.asarray(data["frame_count"], dtype=np.int64),
        landmark_count=np.asarray(data["landmark_count"], dtype=np.int64),
        feature_count=np.asarray(data["feature_count"], dtype=np.int64),
        source_json=np.asarray(data["source_json"]),
        category=np.asarray(data["category"]),
        video_path=np.asarray(data.get("video_path") or ""),
        fps=np.asarray(data.get("fps") if data.get("fps") is not None else np.nan, dtype=np.float32),
    )

    return relative_path.parts[0]

json_paths = []
for category in CATEGORIES:
    json_paths.extend(sorted((JSON_OUTPUT_BASE / category).glob("*.json")))

if not json_paths:
    raise FileNotFoundError(f"No JSON files found in {JSON_OUTPUT_BASE}")

category_counts = Counter()
errors = []

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(convert_json_to_npz, path): path for path in json_paths}

    for future in tqdm(as_completed(futures), total=len(futures), desc="hu_transrac_npz"):
        path = futures[future]
        try:
            category_counts[future.result()] += 1
        except Exception as exc:
            errors.append((str(path.relative_to(JSON_OUTPUT_BASE)), str(exc)))

print("completed", sum(category_counts.values()))
print("category_counts", dict(category_counts))
print("errors", len(errors))

if errors:
    print("first_errors")
    for path, error in errors[:10]:
        print("-", path, error)

print("output", NPZ_OUTPUT_BASE)


## 13. Validate Hu/TransRAC NPZ Files

This cell validates the generated NPZ files, including array names, shapes, finite values, and label consistency.


In [ ]:
BASE = NPZ_OUTPUT_BASE
EXPECTED_KEYS = {
    "pose_sequence", "normalized_pose_sequence", "motion_signal", "density_map",
    "time_points", "rep_count", "category_index", "frame_count", "landmark_count",
    "feature_count", "source_json", "category", "video_path", "fps",
}

category_counts = Counter()
rep_counts = Counter()
errors = []
samples = []

for path in sorted(BASE.rglob("*.npz")):
    rel = path.relative_to(BASE)
    category = rel.parts[0] if len(rel.parts) > 1 else "unknown"
    category_counts[category] += 1

    try:
        with np.load(path, allow_pickle=False) as data:
            keys = set(data.files)
            missing = EXPECTED_KEYS - keys
            if missing:
                errors.append((str(rel), f"missing keys {sorted(missing)}"))
                continue

            pose = data["pose_sequence"]
            normalized = data["normalized_pose_sequence"]
            motion = data["motion_signal"]
            density = data["density_map"]
            time_points = data["time_points"]
            rep_count = int(data["rep_count"])
            density_sum = float(density.sum())

            if pose.shape != (EXPECTED_FRAMES, EXPECTED_LANDMARKS, FEATURES):
                errors.append((str(rel), f"pose_sequence shape {pose.shape}"))
            if normalized.shape != (EXPECTED_FRAMES, EXPECTED_LANDMARKS, FEATURES):
                errors.append((str(rel), f"normalized_pose_sequence shape {normalized.shape}"))
            if motion.shape != (EXPECTED_FRAMES,):
                errors.append((str(rel), f"motion_signal shape {motion.shape}"))
            if density.shape != (EXPECTED_FRAMES,):
                errors.append((str(rel), f"density_map shape {density.shape}"))
            if not np.all(np.isfinite(pose)):
                errors.append((str(rel), "pose_sequence non-finite"))
            if not np.all(np.isfinite(normalized)):
                errors.append((str(rel), "normalized_pose_sequence non-finite"))
            if not np.all(np.isfinite(motion)):
                errors.append((str(rel), "motion_signal non-finite"))
            if not np.all(np.isfinite(density)):
                errors.append((str(rel), "density_map non-finite"))
            if rep_count != len(time_points):
                errors.append((str(rel), "rep_count does not match time_points length"))
            if abs(density_sum - rep_count) > 1e-4:
                errors.append((str(rel), f"density_sum {density_sum} != rep_count {rep_count}"))
            if np.any(time_points < 0) or np.any(time_points >= EXPECTED_FRAMES):
                errors.append((str(rel), "time_points out of range"))

            rep_counts[rep_count] += 1
            if len(samples) < 2:
                samples.append({
                    "path": str(rel),
                    "pose_shape": list(pose.shape),
                    "normalized_shape": list(normalized.shape),
                    "motion_shape": list(motion.shape),
                    "density_shape": list(density.shape),
                    "time_points": time_points.tolist(),
                    "rep_count": rep_count,
                    "density_sum": density_sum,
                })
    except Exception as exc:
        errors.append((str(rel), f"npz_load_error: {exc}"))

print("=" * 78)
print("HU TRANSRAC NPZ VALIDATION REPORT".center(78))
print("=" * 78)
print(f"{'Category':<20} | {'Files':<6}")
print("-" * 78)
for category in CATEGORIES:
    print(f"{category:<20} | {category_counts[category]:<6}")
print("-" * 78)
print(f"{'TOTAL':<20} | {sum(category_counts.values()):<6}")
print("=" * 78)
print("rep_count distribution:", dict(sorted(rep_counts.items())))
print("errors:", len(errors))
print(
    "VALIDATION_OK:",
    sum(category_counts.values()) == 1000
    and all(category_counts[category] == 250 for category in CATEGORIES)
    and len(errors) == 0,
)
print("SAMPLES")
for sample in samples:
    print(sample)
if errors:
    print("FIRST_ERRORS")
    for path, error in errors[:25]:
        print("-", path, error)
